# 두 결합모델의 동일 임계값 비교

실제 전체 Test/Validation 결과를 조회합니다. 이 표를 보고 새 Test 임계값을 고르는 작업은 독립 검증으로 간주하지 않습니다.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, HTML
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "gnn_hgb_fusion"]
ROOT = next((p.resolve() for p in candidates if (p / "PACKAGE.json").exists()
             and json.loads((p / "PACKAGE.json").read_text())["package"] == "gnn_hgb_fusion"), None)
if ROOT is None:
    raise FileNotFoundError("압축을 풀고 해당 패키지의 notebooks 폴더에서 실행하세요.")
sys.path.insert(0, str(ROOT / "src"))
print("패키지:", ROOT)


In [ ]:
scores = pd.read_csv(ROOT / 'results/threshold_comparison/임계값별_전체지표.csv')
SCOPE = 'test'
THRESHOLD = 0.5
selected = scores[(scores.scope == SCOPE) & ((scores.threshold_probability - THRESHOLD).abs() < 1e-12)]
display(selected[['model','threshold_probability','precision','recall','F1','AP','FP','FN']])

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for model, group in scores[scores.scope == SCOPE].groupby('model'):
    group = group.sort_values('threshold_probability')
    for ax, metric in zip(axes, ['precision', 'recall', 'F1']):
        ax.plot(group.threshold_probability, 100 * group[metric], label=model)
        ax.set(xlabel='Probability threshold', ylabel=metric + ' (%)', ylim=(0, 100))
        ax.grid(alpha=.25)
axes[-1].legend(fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
display(pd.read_csv(ROOT / 'results/threshold_comparison/기본값_검증선택값_비교.csv'))

브라우저용 대화형 표: [두모델_임계값비교.html](../results/threshold_comparison/두모델_임계값비교.html). 임계값별 AP는 같은 점수의 순위 지표이므로 값이 바뀌지 않습니다.